# Извлечение текстовых эмбеддингов Qwen3-Embedding-0.6B

Прогоняет текст (title + description) 144313 объявлений через Qwen3-Embedding-0.6B
и сохраняет эмбеддинги в parquet. Сравниваем с TF-IDF: бьёт ли семантика char-ngram

Вход: text_for_emb.parquet (item_id, title, description), загружен как Kaggle Dataset
Выход: /kaggle/working/text_emb_qwen3.parquet (item_id + 1024 признака)

Модель Apache-2.0, ungated - логин в HF НЕ нужен, только Internet ON для скачивания весов
Запуск: Accelerator GPU T4 x2, Internet ON, Run All

In [ ]:
!pip install -q -U sentence-transformers transformers

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
EMB_DIM = 1024            # дефолтная размерность Qwen3-Embedding-0.6B
BATCH_SIZE = 64
MAX_SEQ_LEN = 512         # описания объявлений короткие, 512 токенов с запасом
INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
OUT_PARQUET = WORK_DIR / 'text_emb_qwen3.parquet'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPECTED_ROWS = 144313

## Шаг 1: загрузить текст

In [ ]:
def find_text_parquet():
    """найти загруженный text_for_emb.parquet в /kaggle/input"""
    hit = next(INPUT_DIR.rglob('text_for_emb.parquet'), None)
    if hit is None:
        hit = next(INPUT_DIR.rglob('*.parquet'))
    return hit

In [ ]:
def build_texts(df):
    """склеить title + description ровно как в TF-IDF-бейзлайне, чтобы сравнение было честным"""
    title = df['title'].fillna('')
    desc = df['description'].fillna('')
    return (title + ' ' + desc).str.strip().tolist()

In [ ]:
src = find_text_parquet()
data = pd.read_parquet(src, columns=['item_id', 'title', 'description'])
item_ids = data['item_id'].to_numpy()
texts = build_texts(data)
print('текстов:', len(texts), 'из', src)
print('пример:', texts[0][:120])

## Шаг 2: модель

In [ ]:
def load_model():
    """Qwen3-Embedding на gpu в fp16, last-token pooling и нормализацию делает сам"""
    model = SentenceTransformer(MODEL_NAME, model_kwargs={'torch_dtype': torch.float16}, device=DEVICE)
    model.max_seq_length = MAX_SEQ_LEN
    return model

In [ ]:
model = load_model()
print('модель загружена, max_seq_length =', model.max_seq_length)

## Шаг 3: прогон

In [ ]:
def encode_all(model, texts):
    """эмбеддинги всех текстов: батчами, с нормализацией, как float32 numpy"""
    embs = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    return embs.astype('float32')

In [ ]:
embs = encode_all(model, texts)
print('эмбеддинги:', embs.shape)

## Шаг 4: собрать и сохранить

In [ ]:
def to_dataframe(item_ids, embs):
    """собрать таблицу item_id плюс столбцы признаков"""
    columns = [f'emb_{i}' for i in range(EMB_DIM)]
    df = pd.DataFrame(embs, columns=columns)
    df.insert(0, 'item_id', item_ids)   # item_id это строка-uuid, в int не приводим
    return df

In [ ]:
emb_df = to_dataframe(item_ids, embs)
emb_df.to_parquet(OUT_PARQUET, index=False)
print('сохранено в', OUT_PARQUET)

## Шаг 5: проверки

In [ ]:
assert len(emb_df) == EXPECTED_ROWS, f'строк {len(emb_df)} вместо {EXPECTED_ROWS}'
assert emb_df['item_id'].nunique() == EXPECTED_ROWS, 'item_id не уникален'
assert emb_df.isna().sum().sum() == 0, 'есть NaN'
assert emb_df.shape[1] == EMB_DIM + 1, 'неверное число столбцов'
print('строк:', len(emb_df))
print('признаков:', EMB_DIM)
print('OK')

## Что скачать

Скачай text_emb_qwen3.parquet из панели Output справа

Локально положи в training/artifacts/embeddings/text_emb_qwen3.parquet - это вход для сравнения с TF-IDF